В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [31]:
import pandas as pd
import numpy as np
from numpy.random import default_rng
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from typing import Tuple


def split_train_val(df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target_col])
    return train_df, val_df


def separate_inputs_targets(df: pd.DataFrame, input_cols: list, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets

In [2]:
raw_df = pd.read_csv('bank_customer/train.csv')

target_col = "Exited"
default_drop_cols = ["id", "CustomerId", "Surname", target_col]

input_cols = [c for c in raw_df.columns if c not in default_drop_cols ]

In [4]:
train_df, val_df = split_train_val(raw_df, target_col)
X_train, y_train = separate_inputs_targets(train_df, input_cols, target_col)
X_val, y_val = separate_inputs_targets(val_df, input_cols, target_col)

1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [9]:
cat_cols = X_train.select_dtypes(include='object').columns

In [10]:
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [34]:
xgb_clf = XGBClassifier(
    max_depth=3,
    n_estimators=10,
    enable_categorical=True,  # для категорійних ознак
    missing=np.nan,  # явне вказування пропущених значень
    random_state=42
)

xgb_clf.fit(X_train, y_train)

train_pred = xgb_clf.predict(X_train)
val_pred = xgb_clf.predict(X_val)

print(classification_report(y_train, train_pred, digits=4))
print(classification_report(y_val, val_pred, digits=4))

              precision    recall  f1-score   support

         0.0     0.9146    0.9639    0.9386      9558
         1.0     0.8210    0.6478    0.7242      2442

    accuracy                         0.8996     12000
   macro avg     0.8678    0.8059    0.8314     12000
weighted avg     0.8956    0.8996    0.8950     12000

              precision    recall  f1-score   support

         0.0     0.9137    0.9569    0.9348      2390
         1.0     0.7928    0.6459    0.7118       610

    accuracy                         0.8937      3000
   macro avg     0.8532    0.8014    0.8233      3000
weighted avg     0.8891    0.8937    0.8895      3000



In [36]:
train_probs = xgb_clf.predict_proba(X_train)[:,1]
val_probs = xgb_clf.predict_proba(X_val)[:,1]

print("ROC-AUC XGB Train:", roc_auc_score(y_train, train_probs))
print("ROC-AUC XGB Val:", roc_auc_score(y_val, val_probs))

ROC-AUC XGB Train: 0.9329558971743529
ROC-AUC XGB Val: 0.9326171205158104


3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [58]:
def objective(params):
    clf = xgb.XGBClassifier(
    n_estimators=int(params['n_estimators']),
        learning_rate=params['learning_rate'],
        max_depth=int(params['max_depth']),
        min_child_weight=params['min_child_weight'],  # Мінімальна сума ваг всіх вибірок, необхідна в кінцевому вузлі
        subsample=params['subsample'],  # Частка вибірок, що використовуються для побудови кожного дерева
        colsample_bytree=params['colsample_bytree'],  # Частка ознак, що використовуються при побудові кожного дерева
        gamma=params['gamma'],  # Мінімальне зменшення втрат, необхідне для виконання поділу
        reg_alpha=params['reg_alpha'],  # Параметр регуляризації L1 (Lasso)
        reg_lambda=params['reg_lambda'],  # Параметр регуляризації L2 (Ridge)
        enable_categorical=True,
        missing=np.nan,
        early_stopping_rounds=20,
        random_state=42,
        eval_metric='auc'
    )

    clf.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False)

    val_probs = clf.predict_proba(X_val)[:,1]
    roc_auc = roc_auc_score(y_val, val_probs)

    return {'loss': -roc_auc, 'status': STATUS_OK}

# Простір гіперпараметрів
space = {
    'n_estimators': hp.quniform('n_estimators', 50, 500, 25),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'gamma': hp.uniform('gamma', 0, 0.5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1)
}

# Оптимізація
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=20, trials=trials, rstate=default_rng(42))

# Перетворення значень гіперпараметрів у кінцеві типи
best['n_estimators'] = int(best['n_estimators'])
best['max_depth'] = int(best['max_depth'])
best['min_child_weight'] = int(best['min_child_weight'])

print("Найкращі гіперпараметри: ", best)

# Навчання фінальної моделі з найкращими гіперпараметрами
final_clf = xgb.XGBClassifier(
    n_estimators=best['n_estimators'],
    learning_rate=best['learning_rate'],
    max_depth=best['max_depth'],
    min_child_weight=best['min_child_weight'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    gamma=best['gamma'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    enable_categorical=True,
    missing=np.nan,
    early_stopping_rounds=20,
    random_state=42,
    eval_metric='auc'
)

final_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)],
    verbose=False)
final_val_probs = final_clf.predict_proba(X_val)[:,1]
final_train_probs = final_clf.predict_proba(X_train)[:,1]
final_train_roc_auc = roc_auc_score(y_train, final_train_probs)
final_roc_auc = roc_auc_score(y_val, final_val_probs)
print("ROC AUC на тренувальній вибірці: {:.4f}".format(final_train_roc_auc))
print("ROC AUC на валідаційній вибірці: {:.4f}".format(final_roc_auc))

100%|██████████| 20/20 [00:02<00:00,  7.30trial/s, best loss: -0.9381799163179917]
Найкращі гіперпараметри:  {'colsample_bytree': np.float64(0.670936318586182), 'gamma': np.float64(0.002744016627414303), 'learning_rate': np.float64(0.2666157724963738), 'max_depth': 4, 'min_child_weight': 6, 'n_estimators': 75, 'reg_alpha': np.float64(0.8609304819134347), 'reg_lambda': np.float64(0.6082491621711366), 'subsample': np.float64(0.6502106773952542)}
ROC AUC на тренувальній вибірці: 0.9432
ROC AUC на валідаційній вибірці: 0.9382


In [53]:
test_df = pd.read_csv('bank_customer/test.csv')

for col in cat_cols:
    test_df[col] = test_df[col].astype('category')

In [54]:
test_df

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,15000,15594796.0,Chu,584.0,Germany,Male,30.0,2.0,146053.66,1.0,1.0,1.0,157891.86
1,15001,15642821.0,Mazzi,551.0,France,Male,39.0,5.0,0.00,2.0,1.0,1.0,67431.28
2,15002,15716284.0,Onyekachi,706.0,France,Male,43.0,8.0,0.00,2.0,1.0,0.0,156768.45
3,15003,15785078.0,Martin,717.0,Spain,Male,45.0,3.0,0.00,1.0,1.0,1.0,166909.87
4,15004,15662955.0,Kenechukwu,592.0,Spain,Male,43.0,8.0,0.00,2.0,1.0,1.0,143681.97
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,24995,15642997.0,Chukwumaobim,639.0,Spain,Male,38.0,10.0,0.00,2.0,1.0,1.0,49637.65
9996,24996,15739271.0,Clements,678.0,Spain,Male,39.0,9.0,0.00,2.0,1.0,1.0,142513.50
9997,24997,15756743.0,Chidiebere,774.0,France,Male,30.0,9.0,0.00,2.0,1.0,0.0,4861.72
9998,24998,15680167.0,Yermakova,595.0,France,Male,38.0,6.0,144875.79,1.0,1.0,0.0,126469.09


In [55]:
X_test = test_df[input_cols].copy()

In [56]:
final_test_probs = final_clf.predict_proba(X_test)[:,1]

In [68]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Exited': final_test_probs
})
submission.to_csv('bank_customer/submission_xgb.csv', index=False)

In [ ]:
# Модель хороша, добре генералізує (ROC AUC на тренувальній вибірці: 0.9432, ROC AUC на валідаційній вибірці: 0.9382). Модель стала трохи краще за модель з п.2 (там ROC-AUC: 0.93261).

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [59]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


In [60]:
cat_feature_indexes = [X_train.columns.get_loc(col) for col in cat_cols]

In [61]:
cat_feature_indexes

[1, 2]

In [62]:
lgb_clf = lgb.LGBMClassifier(
    max_depth=3,
    n_estimators=50,
    learning_rate=0.1,
    cat_feature=cat_feature_indexes,  # для автоматичного розпізнавання категорійних ознак
    missing=np.nan,  # явне вказування пропущених значень
)

lgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)])

train_pred = lgb_clf.predict(X_train)
val_pred = lgb_clf.predict(X_val)

print(classification_report(y_train, train_pred, digits=4))
print(classification_report(y_val, val_pred, digits=4))

[LightGBM] [Warning] Unknown parameter: missing
[LightGBM] [Warning] Unknown parameter: missing
[LightGBM] [Warning] categorical_feature is set with cat_feature=1,2, categorical_column=1,2 will be ignored. Current value: categorical_feature=1,2
[LightGBM] [Info] Number of positive: 2442, number of negative: 9558
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000585 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 843
[LightGBM] [Info] Number of data points in the train set: 12000, number of used features: 10
[LightGBM] [Warning] Unknown parameter: missing
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203500 -> initscore=-1.364561
[LightGBM] [Info] Start training from score -1.364561
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

In [63]:
lgb_final_val_probs = lgb_clf.predict_proba(X_val)[:,1]
lgb_final_train_probs = lgb_clf.predict_proba(X_train)[:,1]
lgb_final_train_roc_auc = roc_auc_score(y_train, lgb_final_train_probs)
lgb_final_roc_auc = roc_auc_score(y_val, lgb_final_val_probs)
print("ROC AUC lgb на тренувальній вибірці: {:.4f}".format(lgb_final_train_roc_auc))
print("ROC AUC lgb на валідаційній вибірці: {:.4f}".format(lgb_final_roc_auc))

[LightGBM] [Warning] Unknown parameter: missing
[LightGBM] [Warning] Unknown parameter: missing
ROC AUC lgb на тренувальній вибірці: 0.9388
ROC AUC lgb на валідаційній вибірці: 0.9362


In [65]:
lgb_final_test_probs = lgb_clf.predict_proba(X_test)[:,1]

[LightGBM] [Warning] Unknown parameter: missing


In [67]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Exited': lgb_final_test_probs
})
submission.to_csv('bank_customer/submission_lgb.csv', index=False)

In [ ]:
# Модель хороша без ознак high bias/high variance. ROC AUC на тренувальній вибірці: 0.9388, на валідаційній вибірці: 0.9362. По якості модель не перевершила XGBoost, але різниця мінімальна.

5. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `LightGBM` з лекції знайдіть оптимальні значення гіперпараметрів `LightGBM` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **10**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. І тут ми також ставимо loss - негативне значення AUROC, як і при пошуці гіперпараметрів для XGBoost. До речі, можна спробувати написати код так, аби в objective передавати лише модель і не писати схожий код двічі :)

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_lgb_clf` модель `LightGBM` з найкращими гіперпараметрами
    - навчіть модель `final_lgb_clf`
    - оцініть якість моделі `final_lgb_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (4) цього завдання?

In [77]:
def objective(params):
    clf = lgb.LGBMClassifier(
        n_estimators=int(params['n_estimators']),  # Кількість дерев у ансамблі (кількість ітерацій бустингу)
        learning_rate=params['learning_rate'],  # Коефіцієнт, на який зменшується внесок кожного доданого дерева
        max_depth=int(params['max_depth']),  # Максимальна глибина кожного дерева
        num_leaves=int(params['num_leaves']),  # Максимальна кількість листків, що дозволяємо кожному дереву мати.
        min_child_weight=params['min_child_weight'],  # Мінімальна сума ваг всіх вибірок, необхідна в кінцевому вузлі
        subsample=params['subsample'],  # Частка вибірок, що використовуються для побудови кожного дерева
        colsample_bytree=params['colsample_bytree'],  # Частка ознак, що використовуються при побудові кожного дерева
        reg_alpha=params['reg_alpha'],  # Параметр регуляризації L1 (Lasso)
        reg_lambda=params['reg_lambda'],  # Параметр регуляризації L2 (Ridge)
        min_split_gain=params['min_split_gain'],  # Мінімальне зменшення втрат, необхідне для виконання поділу
        cat_feature=cat_feature_indexes,  # Індекси категорійних ознак
        verbose=-1
    )

    clf.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    val_probs = clf.predict_proba(X_val)[:,1]
    roc_auc = roc_auc_score(y_val, val_probs)

    return {'loss': -roc_auc, 'status': STATUS_OK}

# Простір гіперпараметрів
space = {
    'n_estimators': hp.quniform('n_estimators', 50, 500, 25),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'num_leaves': hp.quniform('num_leaves', 20, 150, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1),
    'min_split_gain': hp.uniform('min_split_gain', 0, 0.1)  # додано мінімальне зменшення втрат для поділу
}

# Оптимізація
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=10, trials=trials, rstate=default_rng(42))

# Перетворення значень гіперпараметрів у кінцеві типи
best['n_estimators'] = int(best['n_estimators'])
best['max_depth'] = int(best['max_depth'])
best['num_leaves'] = int(best['num_leaves'])
best['min_child_weight'] = int(best['min_child_weight'])

print("Найкращі гіперпараметри: ", best)

# Навчання фінальної моделі з найкращими гіперпараметрами
final_clf = lgb.LGBMClassifier(
    n_estimators=best['n_estimators'],
    learning_rate=best['learning_rate'],
    max_depth=best['max_depth'],
    num_leaves=best['num_leaves'],
    min_child_weight=best['min_child_weight'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    min_split_gain=best['min_split_gain'],
    cat_feature=cat_feature_indexes,
    missing=np.nan,
    verbose=-1
)

final_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)])
final_lgb_pred = final_clf.predict(X_val)
final_lgb_train_probs = final_clf.predict_proba(X_train)[:,1]
final_lgb_val_probs = final_clf.predict_proba(X_val)[:,1]
final_lgb_roc_auc_train = roc_auc_score(y_train, final_lgb_train_probs)
final_lgb_roc_auc = roc_auc_score(y_val, final_lgb_val_probs)

print("ROC AUC Final LGB на тренувальній вибірці: {:.4f}".format(final_lgb_roc_auc_train))
print("ROC AUC Final LGB на валідаційній вибірці: {:.4f}".format(final_lgb_roc_auc))

print(classification_report(y_val, final_lgb_pred, digits=4))

100%|██████████| 10/10 [00:13<00:00,  1.36s/trial, best loss: -0.9354835722614719]
Найкращі гіперпараметри:  {'colsample_bytree': np.float64(0.6043143849848778), 'learning_rate': np.float64(0.10914306477998553), 'max_depth': 3, 'min_child_weight': 9, 'min_split_gain': np.float64(0.007898327983574649), 'n_estimators': 300, 'num_leaves': 28, 'reg_alpha': np.float64(0.8609304819134347), 'reg_lambda': np.float64(0.6082491621711366), 'subsample': np.float64(0.6502106773952542)}
ROC AUC Final LGB на тренувальній вибірці: 0.9512
ROC AUC Final LGB на валідаційній вибірці: 0.9355
              precision    recall  f1-score   support

         0.0     0.9211    0.9573    0.9389      2390
         1.0     0.8023    0.6787    0.7353       610

    accuracy                         0.9007      3000
   macro avg     0.8617    0.8180    0.8371      3000
weighted avg     0.8969    0.9007    0.8975      3000



In [74]:
lgb_best_final_test_probs = final_clf.predict_proba(X_test)[:,1]

6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Exited': lgb_best_final_test_probs
})
submission.to_csv('bank_customer/submission_best_lgb.csv', index=False)

In [ ]:
Your Best Entry!Your most recent submission scored 0.93777, which is an improvement over your previous score of 0.93597. Great job!

In [ ]:
# Після успішного пошуку найкращих гіперпараметрів для LightGBM, ця модель виявилась найкращою на тестових даних (scored 0.93777) попри те, що на валідаційному наборі ROC AUC становив 0.9355. Проте в ціє моделі найкращий F1 score на мінорному класі 0.7353